# Figuras del artículo

Este cuaderno produce las dos figuras del artículo derivado de la tesis, y la tabla que sostiene
la segunda. Vive en `PAPER_BPN\` y no en `CODIGO_VSC\notebooks\` por una razón: **no forma parte
del análisis de la tesis**. La tesis está cerrada en la v84 y sus ocho cuadernos no se tocan. Este
es un noveno cálculo, hecho para el artículo, y conviene que se vea que lo es.

## Qué hace, y en qué orden

1. **Recalcula la descomposición de la banda de 28 a 37 semanas** en los catorce años en que la
   variable `T_GES_AGRU_CIE` existe. El Cuadro 5.27 del manuscrito reporta solo cuatro; aquí salen
   los catorce, que es lo que permite dibujar una figura y no una tabla con cuatro filas.
2. **Comprueba que los cuatro años del manuscrito coinciden.** Si no coincidieran, el cuaderno se
   detiene: significaría que los intermedios no son los que sostienen el documento.
3. **Guarda la tabla** en `salidas\tablas\banda3_descompuesta.csv`, de donde la lee el guion que
   dibuja.
4. **Dibuja las dos figuras** llamando a `LATEX\figuras\generar_figuras.py`.

## Qué NO hace

No reconstruye la matriz, no vuelve a aplicar exclusiones y no estima ningún modelo. Lee
`intermedios\muestra_analitica.parquet`, que es la muestra analítica de 17.376.860 filas que ya
dejaron los cuadernos anteriores, y cuenta. La regla del proyecto se mantiene: **el artículo
depende de la salida de los cuadernos.**

## Qué hace falta para correrlo

Solo `intermedios\muestra_analitica.parquet`. **No hace falta la matriz completa.** El cálculo
necesita dos columnas de gestación y el desenlace, y las tres están en la muestra analítica.

## Cuánto tarda

Un par de minutos. La lectura del parquet es lo único pesado: son diecisiete millones de filas,
de las que se leen cuatro columnas.

## 1. Preparación

Se localizan las tres carpetas que el cuaderno necesita. Se buscan por orden y se usa la primera
que exista, para que el cuaderno funcione tanto en `TESIS\CODIGO_VSC` como en una copia del
repositorio, sin editar rutas a mano.

In [1]:
# Bibliotecas. Ninguna es opcional.
import subprocess          # para llamar al guion que dibuja, al final
import sys                 # para usar el mismo intérprete que corre este cuaderno
import time                # para cronometrar la lectura, que es lo único lento
from pathlib import Path   # rutas independientes del sistema operativo

import numpy as np        # el remuestreo se hace sobre matrices, no sobre DataFrames
import pandas as pd

# ----------------------------------------------------------------------
# Localizar el código de la tesis. Este cuaderno vive en PAPER_BPN, que es
# hermana de CODIGO_VSC y no su hija, de modo que hay que subir un nivel.
# ----------------------------------------------------------------------
AQUI = Path.cwd()
CANDIDATAS_CODIGO = [
    AQUI.parent / "CODIGO_VSC",   # PAPER_BPN esta dentro de TESIS
    AQUI.parent,                  # el cuaderno se movio dentro del arbol del codigo
    AQUI,                         # se copio junto al codigo
]
CODIGO = next((c for c in CANDIDATAS_CODIGO
               if (c / "intermedios").is_dir()), None)
if CODIGO is None:
    raise SystemExit(
        "No encuentro la carpeta del código (la que tiene intermedios\\).\n"
        "Rutas probadas:\n  " + "\n  ".join(str(c) for c in CANDIDATAS_CODIGO)
    )

INTERMEDIOS = CODIGO / "intermedios"
TABLAS      = CODIGO / "salidas" / "tablas"
TABLAS.mkdir(parents=True, exist_ok=True)

# El guion que dibuja vive con las figuras, dentro del proyecto LaTeX. Se
# buscan las dos ubicaciones posibles: la carpeta de trabajo PAPER_BPN, y la
# del repositorio publicado, donde los cuadernos cuelgan de notebooks\ y el
# proyecto LaTeX no viaja. Si no esta, el cuaderno calcula y guarda igual:
# lo unico que no ocurre es el dibujo.
CANDIDATAS_GUION = [
    AQUI / "LATEX" / "figuras" / "generar_figuras.py",          # PAPER_BPN
    AQUI.parent / "PAPER_BPN" / "LATEX" / "figuras" / "generar_figuras.py",
    AQUI.parent.parent / "PAPER_BPN" / "LATEX" / "figuras" / "generar_figuras.py",
]
DIBUJA = next((g for g in CANDIDATAS_GUION if g.exists()), CANDIDATAS_GUION[0])

MUESTRA = INTERMEDIOS / "muestra_analitica.parquet"

print(f"código      : {CODIGO}")
print(f"intermedios : {INTERMEDIOS}")
print(f"tablas      : {TABLAS}")
print(f"guion       : {DIBUJA}  {'OK' if DIBUJA.exists() else 'NO ESTA'}")
print()
if MUESTRA.exists():
    print(f"muestra analítica: {MUESTRA.stat().st_size / 1024**2:,.0f} MB")
else:
    raise SystemExit(
        f"Falta {MUESTRA}.\n"
        "Es el intermedio que dejan los cuadernos 01 a 03. Sin él este cuaderno no puede "
        "hacer nada, y no hay forma de reconstruirlo desde aquí."
    )

código      : /sessions/inspiring-laughing-galileo/mnt/TESIS/CODIGO_VSC
intermedios : /sessions/inspiring-laughing-galileo/mnt/TESIS/CODIGO_VSC/intermedios
tablas      : /sessions/inspiring-laughing-galileo/mnt/TESIS/CODIGO_VSC/salidas/tablas
guion       : /sessions/inspiring-laughing-galileo/mnt/TESIS/PAPER_BPN/LATEX/figuras/generar_figuras.py  OK

muestra analítica: 232 MB


**Cómo se lee la salida.** Las cuatro rutas deben apuntar a carpetas que existen, el guion debe
decir `OK`, y la muestra analítica debe pesar del orden de doscientos cuarenta megabytes. Si el
guion dice `NO ESTA`, el cálculo igual se hace y la tabla se guarda: lo único que no ocurrirá es
el dibujo de la última sección.

## 2. La descomposición de la banda

### Por qué hace falta

El certificado de nacido vivo consigna **dos** clasificaciones de la duración de la gestación.
`T_GES` existe en los veintisiete años y su tercera banda va de **28 a 37** semanas, de modo que
el corte de las 37 semanas de la Organización Mundial de la Salud cae *dentro* de ella y mezcla
prematuros con nacimientos que ya son a término. `T_GES_AGRU_CIE` sigue el corte correcto —su
banda 3 va de 28 a **36** y la 4 de 37 a 41— pero solo existe en catorce de los veintisiete años.

Cruzar las dos permite partir la banda gruesa en sus dos componentes y establecer cuál de los dos
es el que crece. Ese es el resultado que sostiene el tercer bloque de los Resultados del artículo.

### La comprobación previa, que no es un adorno

Antes de descomponer se verifica que las dos clasificaciones **coincidan al 100 % fuera de la
banda 3**. Si discreparan también en las bandas 1, 2, 4 y 5, no serían la misma información con
distinto agrupamiento, sino dos variables distintas, y la descomposición no significaría nada.

In [2]:
t0 = time.time()

# Se leen SOLO las cuatro columnas necesarias. Leer las cincuenta y tres sobre
# diecisiete millones de filas consume varios gigabytes sin ninguna ganancia.
gest = pd.read_parquet(MUESTRA, columns=["ANIO", "T_GES", "T_GES_AGRU_CIE", "BPN"])
print(f"leídas {len(gest):,} filas en {time.time() - t0:.0f} s")

# La comparación solo es posible donde existe la segunda clasificación.
mask = gest["T_GES_AGRU_CIE"].notna()

# Se construye un marco reducido y se libera el original de inmediato: sobre
# diecisiete millones de filas, mantener las dos copias agota el intérprete.
#
# Los códigos se guardan como ENTEROS y no como texto. Las dos columnas vienen
# como categóricas con dominios distintos, y pandas no permite compararlas
# directamente; convertirlas a cadena funcionaría, pero una columna de cadenas
# de Python sobre nueve millones de filas consume dos órdenes de magnitud más
# memoria que una de int8.
cruce = pd.DataFrame({
    "ANIO": gest.loc[mask, "ANIO"].to_numpy(dtype="int16"),
    "BPN":  gest.loc[mask, "BPN"].to_numpy(dtype="int8"),
    "ges":  pd.to_numeric(gest.loc[mask, "T_GES"], errors="coerce").to_numpy(dtype="int8"),
    "cie":  pd.to_numeric(gest.loc[mask, "T_GES_AGRU_CIE"], errors="coerce").to_numpy(dtype="int8"),
})
del gest, mask

print(f"registros con las dos clasificaciones: {len(cruce):,}")
print(f"años cubiertos: {cruce['ANIO'].nunique()}  "
      f"({cruce['ANIO'].min()} a {cruce['ANIO'].max()})")

# --- la comprobacion previa -------------------------------------------
otras = cruce[cruce["ges"] != 3]
concordancia = 100 * (otras["ges"] == otras["cie"]).mean()
print(f"\nconcordancia fuera de la banda 3: {concordancia:.2f} %")
assert concordancia > 99.9, (
    "Las dos clasificaciones NO son la misma información con distinto agrupamiento. "
    "La descomposición que sigue no tendría sentido."
)
print("Las dos clasificaciones son la misma variable con distinto agrupamiento.")

leídas 17,376,860 filas en 0 s


registros con las dos clasificaciones: 8,659,594
años cubiertos: 14  (2008 a 2024)



concordancia fuera de la banda 3: 100.00 %
Las dos clasificaciones son la misma variable con distinto agrupamiento.


**Cómo se lee la salida.** La concordancia fuera de la banda 3 debe ser del **100,00 %**. El
número de registros debe rondar los nueve millones y los años cubiertos deben ser **catorce**,
porque 2012, 2013 y 2020 no tienen la variable. Si la comprobación fallara, el cuaderno se
detiene con un error explícito: es preferible a producir una tabla que parece correcta.

## 3. La tabla, año por año

Ahora se reparte, en cada año, la banda de 28 a 37 semanas entre sus dos componentes. Todos los
porcentajes están calculados **sobre el total de nacimientos del año**, salvo la última columna,
que es sobre el total de la banda.

In [3]:
filas = []
for anio in sorted(cruce["ANIO"].unique()):
    x = cruce[cruce["ANIO"] == anio]
    n = len(x)
    en_banda3 = x["ges"] == 3
    pretermino = int((en_banda3 & (x["cie"] == 3)).sum())   # 28 a 36: pretérmino verdadero
    semana37   = int((en_banda3 & (x["cie"] == 4)).sum())   # 37: ya a término según la OMS
    filas.append({
        "anio":        int(anio),
        "n":           n,
        "banda_28_37": 100 * en_banda3.mean(),
        "pretermino":  100 * pretermino / n,
        "semana_37":   100 * semana37 / n,
        "pct_sem37":   100 * semana37 / en_banda3.sum(),
    })

banda3 = pd.DataFrame(filas)
print(banda3.to_string(index=False, float_format=lambda v: f"{v:,.2f}"))

# El reparto del crecimiento, que es el resultado del apartado.
sube_pre = banda3["pretermino"].iloc[-1] - banda3["pretermino"].iloc[0]
sube_s37 = banda3["semana_37"].iloc[-1]  - banda3["semana_37"].iloc[0]
total    = sube_pre + sube_s37
a0, a1 = int(banda3["anio"].iloc[0]), int(banda3["anio"].iloc[-1])

print(f"\nCRECIMIENTO {a0} a {a1}, en puntos porcentuales")
print(f"  pretérmino verdadero (28 a 36) ... {sube_pre:+6.2f}   ({100 * sube_pre / total:4.0f} %)")
print(f"  semana 37 ....................... {sube_s37:+6.2f}   ({100 * sube_s37 / total:4.0f} %)")
print(f"  total de la banda ............... {total:+6.2f}")

 anio      n  banda_28_37  pretermino  semana_37  pct_sem37
 2008 696179        18.25        8.76       9.49      51.98
 2009 683775        17.96        8.64       9.32      51.87
 2010 643656        17.93        8.41       9.53      53.12
 2011 658658        18.08        8.61       9.46      52.36
 2014 663940        19.12        8.90      10.22      53.45
 2015 655866        19.35        8.97      10.38      53.63
 2016 641076        19.74        9.08      10.65      53.98
 2017 649493        19.84        9.02      10.82      54.52
 2018 637947        20.20        9.13      11.06      54.78
 2019 625040        20.96        9.30      11.66      55.62
 2021 598057        23.45       10.28      13.17      56.15
 2022 561858        24.98       10.57      14.42      57.71
 2023 504150        26.41       10.66      15.76      59.65
 2024 439899        27.32       10.65      16.67      61.03

CRECIMIENTO 2008 a 2024, en puntos porcentuales
  pretérmino verdadero (28 a 36) ...  +1.88   (  21

**Cómo se lee la salida.** La columna `pretermino` se mantiene casi plana, del 8,76 al 10,65 %, y
la de `semana_37` es la que se dispara, del 9,49 al 16,67 %. El reparto del final debe dar **21 %
y 79 %**. Ese 79 % es la cifra que el artículo y el manuscrito publican, y es el argumento de que
el desplazamiento entre bandas **no es un aumento de la prematuridad**.

La columna `pct_sem37` dice qué fracción de la banda gruesa es, cada año, semana 37: pasa del
52 % en 2008 al 61 % en 2024. La banda no solo crece, sino que se vuelve cada vez menos
representativa de la prematuridad que su nombre sugiere.

## 4. Comprobación contra el manuscrito

Las cifras recién calculadas deben coincidir con las del Cuadro 5.27 de la tesis. Si no
coincidieran, el intermedio no sería el que sostiene el documento, y la figura del artículo
estaría contando algo distinto de lo que dice el texto.

Se comprueba con dos decimales, que es la precisión con que el cuadro las publica.

In [4]:
# Cuadro 5.27 del manuscrito, transcrito. Es la referencia contra la que se compara.
MANUSCRITO = {
    2008: (18.25, 8.76,  9.49),
    2014: (19.12, 8.90, 10.22),
    2019: (20.96, 9.30, 11.66),
    2024: (27.32, 10.65, 16.67),
}

print(f"{'año':>6} {'columna':<14} {'cuaderno':>10} {'manuscrito':>12}   estado")
print("-" * 58)
discrepancias = 0
for anio, (m_banda, m_pre, m_37) in MANUSCRITO.items():
    f = banda3[banda3["anio"] == anio]
    if f.empty:
        print(f"{anio:>6} {'—':<14} {'AUSENTE':>10}")
        discrepancias += 1
        continue
    f = f.iloc[0]
    for etiqueta, calc, publ in [
        ("banda 28 a 37", f["banda_28_37"], m_banda),
        ("pretérmino",    f["pretermino"],  m_pre),
        ("semana 37",     f["semana_37"],   m_37),
    ]:
        ok = round(calc, 2) == publ
        discrepancias += (not ok)
        print(f"{anio:>6} {etiqueta:<14} {calc:>10.2f} {publ:>12.2f}   {'coincide' if ok else 'DISCREPA'}")

print("-" * 58)
if discrepancias:
    raise SystemExit(f"{discrepancias} discrepancias con el manuscrito. NO se guarda la tabla.")
print("Las 12 comprobaciones coinciden con el Cuadro 5.27.")

   año columna          cuaderno   manuscrito   estado
----------------------------------------------------------
  2008 banda 28 a 37       18.25        18.25   coincide
  2008 pretérmino           8.76         8.76   coincide
  2008 semana 37            9.49         9.49   coincide
  2014 banda 28 a 37       19.12        19.12   coincide
  2014 pretérmino           8.90         8.90   coincide
  2014 semana 37           10.22        10.22   coincide
  2019 banda 28 a 37       20.96        20.96   coincide
  2019 pretérmino           9.30         9.30   coincide
  2019 semana 37           11.66        11.66   coincide
  2024 banda 28 a 37       27.32        27.32   coincide
  2024 pretérmino          10.65        10.65   coincide
  2024 semana 37           16.67        16.67   coincide
----------------------------------------------------------
Las 12 comprobaciones coinciden con el Cuadro 5.27.


**Cómo se lee la salida.** Las doce comprobaciones deben decir `coincide`. Si alguna discrepa, el
cuaderno **se detiene y no guarda nada**, que es lo correcto: una tabla que no corresponde al
manuscrito es peor que ninguna tabla, porque nadie se da cuenta.

## 5. Guardar la tabla

La tabla se guarda en la misma carpeta que las demás salidas de la tesis, y no dentro de
`PAPER_BPN`. Es un dato, no un producto del artículo, y el sitio de los datos es `salidas\tablas\`.
El guion que dibuja la busca ahí.

In [5]:
destino = TABLAS / "banda3_descompuesta.csv"
banda3.to_csv(destino, index=False)
print(f"guardado: {destino}")
print(f"  {len(banda3)} años · {destino.stat().st_size:,} bytes")

guardado: /sessions/inspiring-laughing-galileo/mnt/TESIS/CODIGO_VSC/salidas/tablas/banda3_descompuesta.csv
  14 años · 1,243 bytes


## 7. Los intervalos del artículo, por conglomerados municipales

### Por qué este apartado existe

La tesis calcula los intervalos del índice con un bootstrap **paramétrico** sobre la tabla ya
agregada: la composición se extrae de una multinomial con las proporciones observadas y, dentro de
cada nivel educativo, los casos de una binomial con la prevalencia de ese nivel. Es rápido y, con
un ordenamiento discreto, equivale al bootstrap individual.

Los dos esquemas comparten un supuesto: que las observaciones son independientes. **No lo son.**
Los nacimientos están anidados en municipios, y el propio trabajo estima esa dependencia. Un
esquema que la ignore subestima el error estándar.

El artículo reporta, por eso, intervalos obtenidos remuestreando **municipios enteros**. Este
apartado los calcula y, de paso, los compara con los del esquema de la tesis, para que la
diferencia quede documentada y no escondida.

### Qué hace falta

`intermedios\muestra_armonizada.parquet`, que es la que trae `EDUC4_MADRE` y `MUNI_RES` ya
armonizadas. Tarda unos segundos: el remuestreo no se hace sobre los diecisiete millones de filas
sino sobre la tabla de celdas municipio × nivel × año, que tiene unas cien mil.

In [6]:
ARMONIZADA = INTERMEDIOS / "muestra_armonizada.parquet"
if not ARMONIZADA.exists():
    raise SystemExit(f"Falta {ARMONIZADA}. Sin ella este apartado no se puede calcular.")

t0 = time.time()
d = pd.read_parquet(ARMONIZADA, columns=["ANIO", "EDUC4_MADRE", "MUNI_RES", "BPN"])
d = d.dropna(subset=["EDUC4_MADRE", "MUNI_RES", "BPN"])
print(f"filas ordenables: {len(d):,}   ({time.time() - t0:.0f} s)")

# La tabla de celdas. Es sobre ESTO sobre lo que se remuestrea: un municipio
# entero es una fila por cada nivel educativo, y resamplearlo significa tomar
# todas sus filas a la vez. Remuestrear los diecisiete millones de registros
# daria el mismo resultado y tardaria horas.
celdas = (d.groupby(["ANIO", "MUNI_RES", "EDUC4_MADRE"], observed=True)["BPN"]
            .agg(n="size", casos="sum").reset_index())
del d
celdas.to_parquet(INTERMEDIOS / "celdas_muni_educ.parquet", index=False)
print(f"celdas municipio x nivel x año: {len(celdas):,}")

filas ordenables: 16,820,098   (1 s)


celdas municipio x nivel x año: 112,165


**Cómo se lee la salida.** Deben salir del orden de 16,8 millones de filas ordenables —menos que
los 17,4 de la muestra analítica, porque se descartan las que no tienen educación o municipio— y
unas 112.000 celdas.

### El cálculo

Se define el índice una sola vez y se usa en los dos esquemas, para que la comparación no dependa
de dos implementaciones distintas. Dentro de cada año se remuestrean municipios con reemplazo, mil
veces, y se recalcula el índice sobre la suma de los municipios sorteados.

In [7]:
def indice_erreygers(n, casos):
    """
    Índice de concentración de Kakwani con la corrección de Erreygers, sobre
    el vector de niveles ya agregado.

    Es la misma fórmula de src\\desigualdad.py, reescrita aquí sobre arreglos
    de numpy para poder llamarla mil veces por año sin sobrecarga de pandas.
    """
    tot = n.sum()
    if tot == 0:
        return np.nan
    p = n / tot
    y = np.divide(casos, n, out=np.zeros_like(casos, dtype=float), where=n > 0)
    R = np.concatenate([[0.0], np.cumsum(p)[:-1]]) + p / 2   # rango fraccional
    mu = float((p * y).sum())
    if mu <= 0:
        return np.nan
    cov = float((p * y * R).sum() - mu * float((p * R).sum()))
    return 4 * mu * (2 * cov / mu)


# CADA BLOQUE LLEVA SU PROPIA SEMILLA. Si los tres compartieran un mismo
# generador, el resultado dependeria del ORDEN en que se ejecutan las
# celdas: volver a correr una sola cambiaria los intervalos de las otras en
# el tercer decimal. Con una semilla por bloque, cada uno reproduce siempre
# lo mismo, se corra solo o con los demas.
rng = np.random.default_rng(2024)
B = 1000
filas = []
t0 = time.time()
for anio, g in celdas.groupby("ANIO", observed=True):
    niveles = np.sort(g.EDUC4_MADRE.unique())
    munis   = np.sort(g.MUNI_RES.unique())
    im = {m: i for i, m in enumerate(munis)}
    iv = {v: i for i, v in enumerate(niveles)}

    # matrices municipio x nivel: remuestrear es indexar filas y sumar
    N = np.zeros((len(munis), len(niveles)))
    C = np.zeros_like(N)
    N[g.MUNI_RES.map(im), g.EDUC4_MADRE.map(iv)] = g.n.to_numpy()
    C[g.MUNI_RES.map(im), g.EDUC4_MADRE.map(iv)] = g.casos.to_numpy()

    obs = indice_erreygers(N.sum(0), C.sum(0))

    idx = rng.integers(0, len(munis), size=(B, len(munis)))
    rep = np.array([indice_erreygers(N[s].sum(0), C[s].sum(0)) for s in idx])

    filas.append(dict(ANIO=int(anio), n_munis=len(munis), erreygers=obs,
                      er_inf=np.nanpercentile(rep, 2.5),
                      er_sup=np.nanpercentile(rep, 97.5),
                      ee_er=float(np.nanstd(rep, ddof=1))))

cl = pd.DataFrame(filas)
cl.to_csv(TABLAS / "bootstrap_conglomerados_anual.csv", index=False)
print(f"{B} réplicas x {len(cl)} años en {time.time() - t0:.0f} s\n")

cl["contiene_cero"] = np.where((cl.er_inf <= 0) & (cl.er_sup >= 0), "sí", "")
print(cl[["ANIO", "n_munis", "erreygers", "er_inf", "er_sup", "contiene_cero"]]
      .to_string(index=False, float_format=lambda v: f"{v:,.5f}"))
print(f"\naños cuyo intervalo contiene el cero: {(cl.contiene_cero == 'sí').sum()} de {len(cl)}")

1000 réplicas x 27 años en 3 s

 ANIO  n_munis  erreygers   er_inf   er_sup contiene_cero
 1998     1113    0.00123 -0.00683  0.00514            sí
 1999     1128    0.00123 -0.00543  0.00517            sí
 2000     1136    0.00181 -0.00444  0.00549            sí
 2001     1142    0.00160 -0.00450  0.00489            sí
 2002     1146    0.00304 -0.00403  0.00681            sí
 2003     1146    0.00316 -0.00395  0.00712            sí
 2004     1147    0.00442 -0.00268  0.00822            sí
 2005     1148    0.00483 -0.00189  0.00818            sí
 2006     1146    0.00485 -0.00118  0.00815            sí
 2007     1143    0.00469 -0.00222  0.00853            sí
 2008     1118    0.00589 -0.00069  0.00914            sí
 2009     1118    0.00726 -0.00059  0.01207            sí
 2010     1117    0.00675  0.00110  0.00950              
 2011     1117    0.00694  0.00082  0.01005              
 2012     1124    0.00593  0.00006  0.00927              
 2013     1122    0.00388 -0.00208  0.00

**Cómo se lee la salida.** Lo que importa no son los extremos sino la última columna: **22 de los
27 años contienen el cero**. Solo quedan fuera 2010, 2011 y 2012 en positivo y 2023 y 2024 en
negativo. El año 2019, en que la serie de estimaciones puntuales cruza el cero, **sí** lo contiene,
y por eso el artículo no data el cruce en un año concreto.

Las estimaciones puntuales de la columna `erreygers` deben coincidir exactamente con las de
`serie_anual_desigualdad.csv`: el esquema de remuestreo cambia el intervalo, nunca la estimación.

### La comparación con el esquema de la tesis

Para saber cuánto ensancha el agrupamiento hace falta el otro intervalo calculado sobre los mismos
años. Se reproduce aquí el bootstrap paramétrico de la tesis y se ponen los dos lado a lado.

In [8]:
rng = np.random.default_rng(3031)      # semilla propia, ver la nota de arriba
comp = []
for anio, g in celdas.groupby("ANIO", observed=True):
    lv = np.sort(g.EDUC4_MADRE.unique())
    n = g.groupby("EDUC4_MADRE", observed=True).n.sum().reindex(lv).to_numpy(float)
    c = g.groupby("EDUC4_MADRE", observed=True).casos.sum().reindex(lv).to_numpy(float)

    # el esquema de la tesis: multinomial para la composición, binomial para el desenlace
    tot = int(n.sum()); p = n / tot; y = c / n
    rep = np.array([indice_erreygers(nr.astype(float), rng.binomial(nr, y).astype(float))
                    for nr in (rng.multinomial(tot, p) for _ in range(B))])
    comp.append(dict(ANIO=int(anio),
                     par_inf=np.nanpercentile(rep, 2.5),
                     par_sup=np.nanpercentile(rep, 97.5)))

t = cl.merge(pd.DataFrame(comp), on="ANIO")
t["ancho_par"] = t.par_sup - t.par_inf
t["ancho_cl"]  = t.er_sup  - t.er_inf
t["veces"]     = t.ancho_cl / t.ancho_par
t.to_csv(TABLAS / "comparacion_bootstrap.csv", index=False)

print(t[["ANIO", "erreygers", "ancho_par", "ancho_cl", "veces"]]
      .to_string(index=False, float_format=lambda v: f"{v:,.5f}"))
print(f"\nensanchamiento mediano: x{t.veces.median():.1f}  "
      f"(entre x{t.veces.min():.1f} y x{t.veces.max():.1f})")
print(f"contienen el cero -> paramétrico: {((t.par_inf<=0)&(t.par_sup>=0)).sum()} de 27  ·  "
      f"conglomerados: {((t.er_inf<=0)&(t.er_sup>=0)).sum()} de 27")

 ANIO  erreygers  ancho_par  ancho_cl   veces
 1998    0.00123    0.00278   0.01196 4.30391
 1999    0.00123    0.00266   0.01059 3.98145
 2000    0.00181    0.00261   0.00993 3.79895
 2001    0.00160    0.00272   0.00940 3.45427
 2002    0.00304    0.00279   0.01084 3.89168
 2003    0.00316    0.00268   0.01107 4.13155
 2004    0.00442    0.00284   0.01090 3.84208
 2005    0.00483    0.00251   0.01007 4.00784
 2006    0.00485    0.00270   0.00933 3.45373
 2007    0.00469    0.00266   0.01075 4.03936
 2008    0.00589    0.00278   0.00984 3.54099
 2009    0.00726    0.00274   0.01266 4.61850
 2010    0.00675    0.00284   0.00841 2.96176
 2011    0.00694    0.00278   0.00923 3.32435
 2012    0.00593    0.00267   0.00922 3.44997
 2013    0.00388    0.00286   0.01000 3.49261
 2014    0.00526    0.00274   0.01016 3.71012
 2015    0.00161    0.00276   0.00882 3.19094
 2016    0.00044    0.00273   0.00789 2.89097
 2017    0.00012    0.00275   0.00741 2.69695
 2018    0.00203    0.00280   0.01

**Cómo se lee la salida.** El ensanchamiento mediano debe rondar el **×3,6**. Es mucho más de lo
que cabía esperar y es la razón de que la lectura del artículo cambiara: no es un ajuste de
precisión, es un cambio en lo que se puede afirmar.

### La prueba que sí sostiene el hallazgo

Si el año del cruce no es identificable, hay que preguntarse otra cosa: ¿difiere el índice de 2024
del de 1998? Esa es una pregunta sobre una **diferencia**, y se contrasta remuestreando los
municipios **en pareja**, es decir, tomando en cada réplica el mismo conjunto de municipios para
los dos años. Así la comparación no arrastra la variación de qué municipios entran.

In [9]:
def matrices(anio, munis, niveles):
    g = celdas[(celdas.ANIO == anio) & (celdas.MUNI_RES.isin(set(munis)))]
    im = {m: i for i, m in enumerate(munis)}
    iv = {v: i for i, v in enumerate(niveles)}
    N = np.zeros((len(munis), len(niveles))); C = np.zeros_like(N)
    N[g.MUNI_RES.map(im), g.EDUC4_MADRE.map(iv)] = g.n.to_numpy()
    C[g.MUNI_RES.map(im), g.EDUC4_MADRE.map(iv)] = g.casos.to_numpy()
    return N, C

rng = np.random.default_rng(4042)      # semilla propia, ver la nota de arriba
niveles = np.sort(celdas.EDUC4_MADRE.unique())
print("DIFERENCIAS ENTRE AÑOS, municipios remuestreados en pareja\n")
difs = []
for a, b in [(1998, 2024), (2009, 2024), (1998, 2009)]:
    ms = np.sort(np.intersect1d(celdas[celdas.ANIO == a].MUNI_RES.unique(),
                                celdas[celdas.ANIO == b].MUNI_RES.unique()))
    Na, Ca = matrices(a, ms, niveles)
    Nb, Cb = matrices(b, ms, niveles)
    obs = indice_erreygers(Nb.sum(0), Cb.sum(0)) - indice_erreygers(Na.sum(0), Ca.sum(0))
    idx = rng.integers(0, len(ms), size=(B, len(ms)))
    rep = np.array([indice_erreygers(Nb[s].sum(0), Cb[s].sum(0))
                    - indice_erreygers(Na[s].sum(0), Ca[s].sum(0)) for s in idx])
    lo, hi = np.nanpercentile(rep, [2.5, 97.5])
    difs.append(dict(desde=a, hasta=b, municipios=len(ms), diferencia=obs,
                     ic_inf=lo, ic_sup=hi, excluye_cero="sí" if lo * hi > 0 else "no"))
    print(f"  {a} a {b}  ({len(ms):,} municipios en ambos)")
    print(f"     {obs:+.5f}   IC 95 % [{lo:+.5f}, {hi:+.5f}]   "
          f"{'EXCLUYE el cero' if lo*hi > 0 else 'contiene el cero'}\n")

pd.DataFrame(difs).to_csv(TABLAS / "diferencias_pareadas.csv", index=False)
print("guardado: diferencias_pareadas.csv")

DIFERENCIAS ENTRE AÑOS, municipios remuestreados en pareja



  1998 a 2024  (1,088 municipios en ambos)
     -0.00791   IC 95 % [-0.01046, -0.00251]   EXCLUYE el cero



  2009 a 2024  (1,116 municipios en ambos)
     -0.01379   IC 95 % [-0.01688, -0.00841]   EXCLUYE el cero



  1998 a 2009  (1,089 municipios en ambos)
     +0.00596   IC 95 % [+0.00343, +0.00877]   EXCLUYE el cero

guardado: diferencias_pareadas.csv


**Cómo se lee la salida.** Los tres contrastes deben excluir el cero. Ese es el resultado que
sostiene el artículo: **el año del cruce no es datable, pero el cambio entre los extremos sí es
significativo**, y lo es con el esquema más conservador disponible.

Si alguno de los tres pasara a contener el cero, habría que revisar el enunciado del hallazgo antes
de enviar nada.

## 9. El grupo de la semana 37, por dentro

Saber que la banda crece por la semana 37 deja dos preguntas que el propio desplazamiento no
responde, y que deciden si ese grupo merece un tratamiento clínico propio en el artículo:

1. **¿Los nacidos en la semana 37 están naciendo más livianos?** Si la prevalencia de bajo peso
   dentro del grupo es estable, el fenómeno es solo de conteo. Si sube, hay algo más.
2. **¿El crecimiento del grupo es diferencial por nivel educativo?** Si se concentra en las madres
   menos favorecidas, es un fenómeno de desigualdad y pertenece al artículo. Si es uniforme, es un
   cambio general en la práctica obstétrica y pertenece a otro trabajo.

Las dos se contestan con la misma tabla. Hace falta cruzar la clasificación de gestación, que está
en `muestra_analitica.parquet`, con el nivel educativo armonizado, que está en
`muestra_armonizada.parquet`.

In [10]:
ARMONIZADA = INTERMEDIOS / "muestra_armonizada.parquet"

# Las dos tablas tienen las mismas filas en el mismo orden, pero eso NO se
# supone: se comprueba. Pegarlas por posicion sin verificar el orden es la
# clase de error que no lanza excepcion y produce una tabla plausible y falsa.
a = pd.read_parquet(MUESTRA,    columns=["ANIO", "BPN", "T_GES", "T_GES_AGRU_CIE"])
b = pd.read_parquet(ARMONIZADA, columns=["ANIO", "BPN", "EDUC4_MADRE"])
mismo_orden = ((a.ANIO.to_numpy() == b.ANIO.to_numpy()).all()
               and (a.BPN.to_numpy() == b.BPN.to_numpy()).all())
print("las dos tablas están en el mismo orden fila a fila:", mismo_orden)
assert mismo_orden, "No se pueden pegar por posición: el orden difiere."

m = a.T_GES_AGRU_CIE.notna().to_numpy()
g = pd.DataFrame({
    "ANIO": a.ANIO.to_numpy()[m],
    "BPN":  a.BPN.to_numpy()[m],
    "ges":  pd.to_numeric(a.T_GES, errors="coerce").to_numpy()[m],
    "cie":  pd.to_numeric(a.T_GES_AGRU_CIE, errors="coerce").to_numpy()[m],
    "edu":  b.EDUC4_MADRE.to_numpy()[m],
})
del a, b

# Los tres grupos. La semana 37 sola no existe como categoria: es la
# interseccion de "banda 3 de T_GES" con "banda 4 de la CIE".
g["grupo"] = np.select(
    [(g.ges == 3) & (g.cie == 3), (g.ges == 3) & (g.cie == 4), (g.ges == 4)],
    ["28a36", "sem37", "38a41"], default="otro")
g = g[g.grupo != "otro"]
print(f"registros clasificados: {len(g):,}")

NIVEL = {0: "Ninguno", 1: "Primaria", 2: "Secundaria", 3: "Superior"}

# Panel A: prevalencia de bajo peso dentro de cada grupo
A = (g.groupby(["ANIO", "grupo"]).BPN.agg(prev=lambda s: 100 * s.mean())
       .reset_index().pivot(index="ANIO", columns="grupo")["prev"]
       [["28a36", "sem37", "38a41"]])
A.columns = ["prev_28a36", "prev_sem37", "prev_38a41"]

# Panel B: % de semana 37 dentro de cada nivel educativo
B = (g.assign(es37=g.grupo == "sem37")
       .groupby(["ANIO", "edu"]).es37.mean().mul(100).unstack())
B.columns = [f"sem37_{NIVEL.get(c, c)}" for c in B.columns]

sem37 = A.join(B).reset_index()
sem37.to_csv(TABLAS / "semana37_clinico.csv", index=False)
print(sem37.to_string(index=False, float_format=lambda v: f"{v:,.2f}"))

r = sem37.set_index("ANIO")
print("\nPREGUNTA 1 · prevalencia de bajo peso dentro de cada grupo, 2008 a 2024")
for col, et in [("prev_28a36", "28 a 36 semanas"), ("prev_sem37", "semana 37"),
                ("prev_38a41", "38 a 41 semanas")]:
    print(f"  {et:<18} {r.loc[2008, col]:5.2f} -> {r.loc[2024, col]:5.2f} %   "
          f"x{r.loc[2024, col] / r.loc[2008, col]:.2f}")

print("\nPREGUNTA 2 · crecimiento de la semana 37 por nivel educativo, en puntos porcentuales")
cols = [c for c in sem37.columns if c.startswith("sem37_")]
for c in cols:
    print(f"  {c.replace('sem37_', ''):<12} {r.loc[2024, c] - r.loc[2008, c]:+.2f}")
dif = max(r.loc[2024, c] - r.loc[2008, c] for c in cols) - \
      min(r.loc[2024, c] - r.loc[2008, c] for c in cols)
print(f"\n  diferencia entre el nivel que más sube y el que menos: {dif:.2f} pp")

las dos tablas están en el mismo orden fila a fila: True


registros clasificados: 8,600,601


 ANIO  prev_28a36  prev_sem37  prev_38a41  sem37_Ninguno  sem37_Primaria  sem37_Secundaria  sem37_Superior
 2008       54.35       13.36        2.96          10.46            9.20              9.48           10.44
 2009       56.94       14.00        3.03           9.44            8.82              9.28           10.52
 2010       57.93       14.33        3.00          10.44            9.02              9.54           10.43
 2011       58.02       13.96        2.81           9.43            9.09              9.41           10.57
 2014       58.56       12.93        2.22          10.50            9.92             10.06           11.17
 2015       59.13       13.30        2.15          11.24            9.93             10.25           11.29
 2016       59.08       13.66        2.31          11.19           10.33             10.48           11.34
 2017       59.42       14.31        2.20          11.63           10.31             10.55           11.95
 2018       59.17       14.59        

**Cómo se lee la salida, y qué se decidió con ella.**

**Pregunta 1.** Dentro de la semana 37 la prevalencia de bajo peso sube de $13{,}36$ a
$17{,}90$\,%, un $34$\,% relativo. En el mismo período **baja** entre los de 38 a 41 semanas, de
$2{,}96$ a $2{,}21$\,%, y apenas se mueve entre los pretérmino. Es decir: los nacidos en la semana
37 no solo son más, son **más livianos**, y son el único de los tres grupos en que eso ocurre. Un
cambio en la manera de clasificar la edad gestacional no produciría ese patrón.

**Pregunta 2.** El crecimiento es prácticamente uniforme: entre $+6{,}29$ y $+7{,}25$ puntos
porcentuales, y el mayor corresponde a las madres con educación superior. **No es un fenómeno de
desigualdad.**

**La decisión que se tomó con esto**, el 27 de septiembre: la semana 37 **no** abre un apartado
clínico propio en el artículo, porque medida resulta no ser un asunto de equidad, que es el foco de
la revista. Pero los dos hechos **se reportan**, en el Cuadro 4 de Resultados y en un párrafo de la
Discusión, y quedan planteados como investigación futura. Un hallazgo propio no se calla por no
tener dónde ponerlo.

**Y un beneficio colateral.** La pregunta 2 confirma que el mecanismo del artículo opera cambiando
el *peso relativo* de dos subgrupos con gradientes opuestos, y no por un desplazamiento diferencial
entre grupos socioeconómicos. Eso es exactamente lo que el último párrafo de la Discusión pide
verificar a quien aplique el índice en otro contexto.

## 10. Dibujar las dos figuras

Se llama al guion que dibuja. Está separado del cuaderno a propósito: así las figuras se pueden
regenerar sin abrir Jupyter, que es lo que hace falta cuando solo se cambia un color o el tamaño
de una letra.

El guion no calcula nada. Lee `serie_anual_desigualdad.csv` para la Figura 1 y la tabla que se
acaba de guardar para la Figura 2.

In [11]:
if not DIBUJA.exists():
    print(f"No encuentro {DIBUJA}.")
    print("La tabla quedó guardada; las figuras hay que dibujarlas aparte.")
else:
    # Se usa el mismo intérprete que corre este cuaderno, para que use el mismo
    # entorno y no otro que pueda no tener matplotlib.
    r = subprocess.run([sys.executable, str(DIBUJA)],
                       cwd=DIBUJA.parent, capture_output=True, text=True)
    print(r.stdout)
    if r.returncode != 0:
        print("ERROR al dibujar:")
        print(r.stderr)
    else:
        for f in ("fig1_serie.pdf", "fig2_banda3.pdf"):
            p = DIBUJA.parent / f
            print(f"  {f:<20} {p.stat().st_size / 1024:,.0f} kB")

leyendo de: /sessions/inspiring-laughing-galileo/mnt/TESIS/CODIGO_VSC/salidas/tablas
fig1_serie.pdf   27 años · 3 positivos, 2 negativos, 22 sin distinguir de cero
fig2_banda3.pdf  14 años · semana 37 = 79 % del alza

  fig1_serie.pdf       21 kB
  fig2_banda3.pdf      20 kB


**Cómo se lee la salida.** Deben aparecer los dos PDF. La Figura 1 debe decir **27 años** y cruce
en **2019**; la Figura 2, **14 años** y **79 %**. Si la Figura 1 dijera otro año de cruce, o la
Figura 2 otro porcentaje, algo cambió en los intermedios y hay que averiguar qué antes de usar
las figuras.

## Y después

Compilar el artículo: doble clic en `LATEX\compilar.bat`. Las figuras entran solas, porque el
`.tex` las incluye por nombre.

**Lo que este cuaderno no resuelve.** La Figura 1 sale sin su banda de confianza. Añadirla depende
de la decisión sobre el esquema de remuestreo que está pendiente con el director, y es la
séptima del documento de la reunión.